In [1]:
import tensorflow as tf


I0000 00:00:1758328079.332746   60887 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1758328080.419430   60887 cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1758328082.442001   60887 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0-dev20250916


In [3]:
mnist = tf.keras.datasets.mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [4]:
model = tf.keras.models.Sequential([
  tf.keras.layers.Flatten(input_shape=(28, 28)),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dropout(0.2),
  tf.keras.layers.Dense(10)
])

/opt/venv/lib/python3.11/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
W0000 00:00:1758262243.437252   19617 gpu_device.cc:2456] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1758262243.438897   19617 gpu_device.cc:2456] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1758262243.582569   19617 gpu_device.cc:2040] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 29041 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5090, pci bus id: 0000:01:00.0, compute capability: 12.0a


In [5]:
predictions = model(x_train[:1]).numpy()
predictions

array([[ 0.9656706 , -0.24634533,  0.8100711 ,  0.41657758,  0.44612765,
         1.4255755 ,  0.31453574,  0.2251242 , -0.70953727, -0.26091325]],
      dtype=float32)

In [6]:
tf.nn.softmax(predictions).numpy()

array([[0.15652987, 0.04658278, 0.13397422, 0.09039184, 0.09310278,
        0.24793133, 0.08162308, 0.07464179, 0.02931323, 0.04590908]],
      dtype=float32)

In [7]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)


In [8]:
loss_fn(y_train[:1], predictions).numpy()


np.float32(1.3946035)

In [9]:
model.compile(optimizer='adam',
              loss=loss_fn,
              metrics=['accuracy'])

In [10]:
model.fit(x_train, y_train, epochs=5)


Epoch 1/5


I0000 00:00:1758262294.761556   21679 service.cc:158] XLA service 0x7c1dac01a100 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1758262294.761578   21679 service.cc:166]   StreamExecutor device (0): NVIDIA GeForce RTX 5090, Compute Capability 12.0a
I0000 00:00:1758262294.770534   21679 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1758262294.827356   21679 cuda_dnn.cc:463] Loaded cuDNN version 91002


  94/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.5541 - loss: 1.4397

I0000 00:00:1758262295.745678   21679 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9139 - loss: 0.2941
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9574 - loss: 0.1426
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9668 - loss: 0.1075
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9728 - loss: 0.0874
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9768 - loss: 0.0741


In [11]:
model.evaluate(x_test,  y_test, verbose=2)


313/313 - 1s - 3ms/step - accuracy: 0.9770 - loss: 0.0802


[0.08017099648714066, 0.9769999980926514]

In [12]:
probability_model = tf.keras.Sequential([
  model,
  tf.keras.layers.Softmax()
])

In [13]:
probability_model(x_test[:5])

<tf.Tensor: shape=(5, 10), dtype=float32, numpy=
array([[2.4940933e-08, 1.9059034e-09, 2.5163979e-05, 4.4761633e-04,
        1.3469471e-11, 3.8137962e-08, 5.1700198e-14, 9.9948359e-01,
        4.8574407e-07, 4.3176820e-05],
       [3.7220325e-09, 2.4771728e-03, 9.9752027e-01, 1.1831360e-06,
        1.4874742e-16, 3.2846040e-07, 5.7195530e-07, 2.3114884e-14,
        3.5432589e-07, 3.0887036e-14],
       [1.4132314e-06, 9.9832243e-01, 2.8246184e-04, 1.1907430e-05,
        1.2068894e-05, 5.0520961e-05, 1.7401895e-05, 9.6918276e-04,
        3.3110142e-04, 1.5419129e-06],
       [9.9998307e-01, 6.3904730e-09, 6.6640519e-06, 3.6991732e-08,
        9.0373318e-09, 3.2047231e-07, 8.6150703e-06, 2.7400759e-07,
        7.0639148e-09, 1.0348930e-06],
       [1.0011289e-07, 7.0692558e-09, 3.3224097e-07, 1.2702672e-07,
        9.9868935e-01, 1.9921677e-07, 2.7943042e-07, 6.3991181e-05,
        2.1505697e-07, 1.2453349e-03]], dtype=float32)>